In [ ]:
import random
import math
from itertools import combinations

# Reusing setup from Q1/Q2

def load_doc(path):
    with open(path, 'r') as f:
        return f.read().strip()

doc_paths = {
    'D1': 'minhash/D1.txt',
    'D2': 'minhash/D2.txt',
    'D3': 'minhash/D3.txt',
    'D4': 'minhash/D4.txt',
}

docs = {name: load_doc(path) for name, path in doc_paths.items()}

def char_kgrams(text, k):
    return set(text[i:i+k] for i in range(len(text) - k + 1))

def jaccard(set_a, set_b):

    inter = len(set_a & set_b)
    union = len(set_a | set_b)
    return inter / union if union > 0 else 0.0

M = 100_003  # prime > 10,000

def make_hash_funcs(t, m=M, seed=42):

    random.seed(seed)
    p = 2**31 - 1
    params = [(random.randint(1, p-1), random.randint(0, p-1)) for _ in range(t)]
    return [lambda gram, a=a, b=b: (a * hash(gram) + b) % m for a, b in params]

def minhash_signature(gram_set, hash_funcs):
    return [min(h(gram) for gram in gram_set) for h in hash_funcs]


T = 160
TAU = 0.7

print("=" * 65)
print("QUESTION 3A: FINDING BEST (b, r) with b*r = 160, tau = 0.7")
print("=" * 65)
print(f"\nS-curve: f(s) = 1 - (1 - s^b)^r")
print(f"Goal: threshold s* = (1/b)^(1/r) ≈ tau = {TAU}")
print(f"      and maximise steepness |f'(tau)|\n")

def s_curve(s, b, r):
    return 1 - (1 - s**b)**r

def s_curve_derivative(s, b, r):
    # f'(s) = r * b * s^(b-1) * (1 - s^b)^(r-1)
    return r * b * (s**(b-1)) * ((1 - s**b)**(r-1))

best = None
best_score = -1

print(f"{'b':>5} {'r':>5} {'b*r':>6} {'s*':>8} {'f(tau)':>10} {'f\'(tau)':>12}")
print("-" * 55)

for b in range(1, T + 1):

    if T % b == 0:
        r = T // b
        s_star = (1 / b) ** (1 / r)
        f_tau = s_curve(TAU, b, r)
        df_tau = s_curve_derivative(TAU, b, r)

        print(f"  b={b:>3}  r={r:>3}  b*r={b*r:>4}  s*={s_star:.4f}  "
              f"f(τ)={f_tau:.4f}  f'(τ)={df_tau:.4f}")
        
        # Score: maximise derivative at tau (steepness = separation quality)
        score = df_tau
        if score > best_score:
            best_score = score
            best = (b, r, s_star, f_tau, df_tau)

best_b, best_r, best_s, best_f, best_df = best

print(f"""
Best choice: b = {best_b}, r = {best_r}
  Threshold s* = (1/{best_b})^(1/{best_r}) = {best_s:.6f}  (target tau = {TAU})
  f(tau={TAU})  = {best_f:.6f}   (probability of match AT threshold)
  f'(tau={TAU}) = {best_df:.6f}  (steepness / separation quality)
""")

print("=" * 65)
print("QUESTION 3B: PROBABILITY EACH PAIR IS ESTIMATED > tau")
print(f"Using b = {best_b}, r = {best_r}, t = {T}")
print("=" * 65)

# Build 3-gram sets for all docs
grams = {name: char_kgrams(text, 3) for name, text in docs.items()}
doc_names = list(docs.keys())
pairs = list(combinations(doc_names, 2))

# Analytical probability from exact Jaccard
print("\n--- Analytical: P(pair is candidate) = f(J) = 1-(1-J^b)^r ---\n")
print(f"{'Pair':>10}  {'Exact J':>10}  {'P(candidate)':>14}  {'Above tau?':>12}")
print("-" * 55)

for a, b_doc in pairs:

    j = jaccard(grams[a], grams[b_doc])
    p_candidate = s_curve(j, best_b, best_r)
    above = "YES" if j >= TAU else "no"
    print(f"  ({a},{b_doc})    {j:>10.6f}  {p_candidate:>14.6f}  {above:>12}")

# Empirical: running many LSH trials to estimate probability
print("\n--- Empirical: P(candidate pair) over 500 trials ---\n")

TRIALS = 500
candidate_counts = {(a, b_doc): 0 for a, b_doc in pairs}

for trial in range(TRIALS):

    hf = make_hash_funcs(T, M, seed=trial)
    sigs = {name: minhash_signature(grams[name], hf) for name in doc_names}

    # Split each signature into r bands of b rows
    bands = {}
    for name in doc_names:
        sig = sigs[name]
        bands[name] = [tuple(sig[i*best_b:(i+1)*best_b]) for i in range(best_r)]

    # Check each pair: candidate if any band matches
    for a, b_doc in pairs:
        for band_idx in range(best_r):
            if bands[a][band_idx] == bands[b_doc][band_idx]:
                candidate_counts[(a, b_doc)] += 1
                break  # at least one band matched

print(f"{'Pair':>10}  {'Exact J':>10}  {'P(candidate)':>14}  {'Above tau?':>12}")
print("-" * 55)

for a, b_doc in pairs:
    j = jaccard(grams[a], grams[b_doc])
    p_emp = candidate_counts[(a, b_doc)] / TRIALS
    above = "YES" if j >= TAU else "no"
    print(f"  ({a},{b_doc})    {j:>10.6f}  {p_emp:>14.6f}  {above:>12}")


QUESTION 3A: FINDING BEST (b, r) with b*r = 160, tau = 0.7

S-curve: f(s) = 1 - (1 - s^b)^r
Goal: threshold s* = (1/b)^(1/r) ≈ tau = 0.7
      and maximise steepness |f'(tau)|

    b     r    b*r       s*     f(tau)      f'(tau)
-------------------------------------------------------
  b=  1  r=160  b*r= 160  s*=1.0000  f(τ)=1.0000  f'(τ)=0.0000
  b=  2  r= 80  b*r= 160  s*=0.9914  f(τ)=1.0000  f'(τ)=0.0000
  b=  4  r= 40  b*r= 160  s*=0.9659  f(τ)=1.0000  f'(τ)=0.0012
  b=  5  r= 32  b*r= 160  s*=0.9509  f(τ)=0.9972  f'(τ)=0.1280
  b=  8  r= 20  b*r= 160  s*=0.9013  f(τ)=0.6950  f'(τ)=4.2644
  b= 10  r= 16  b*r= 160  s*=0.8660  f(τ)=0.3677  f'(τ)=4.2009
  b= 16  r= 10  b*r= 160  s*=0.7579  f(τ)=0.0327  f'(τ)=0.7372
  b= 20  r=  8  b*r= 160  s*=0.6877  f(τ)=0.0064  f'(τ)=0.1814
  b= 32  r=  5  b*r= 160  s*=0.5000  f(τ)=0.0001  f'(τ)=0.0025
  b= 40  r=  4  b*r= 160  s*=0.3976  f(τ)=0.0000  f'(τ)=0.0001
  b= 80  r=  2  b*r= 160  s*=0.1118  f(τ)=0.0000  f'(τ)=0.0000
  b=160  r=  1  b*r= 1